<a href="https://colab.research.google.com/github/moushumipriya/Amazon-Customer-Reviews-Sentiment-Analysis-NLP/blob/main/Amazon_Customer_Reviews_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [5]:
# Replace with your dataset path in Drive
file_path = "/content/drive/MyDrive/GTSRB/Reviews.csv"
df = pd.read_csv(file_path)

df.head()


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [8]:
# Lowercase
df['Text'] = df['Text'].str.lower()

# Remove punctuation
df['Text'] = df['Text'].str.replace('[^\w\s]', '', regex=True)

# Remove stopwords
stop_words = set(stopwords.words('english'))
df['Text'] = df['Text'].apply(lambda x: ' '.join([w for w in x.split() if w not in stop_words]))

# Lemmatization
lemmatizer = WordNetLemmatizer()
df['Text'] = df['Text'].apply(lambda x: ' '.join([lemmatizer.lemmatize(w) for w in x.split()]))


In [10]:
def sentiment_label(rating):
    if rating >= 4:
        return "Positive"
    elif rating == 3:
        return "Neutral"
    else:
        return "Negative"

df['Sentiment'] = df['Score'].apply(sentiment_label)
df['Sentiment'].value_counts()

,count
Sentiment,
Positive,443777
Negative,82037
Neutral,42640


In [11]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['Text'])

le = LabelEncoder()
y = le.fit_transform(df['Sentiment'])


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [13]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)


In [14]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


Logistic Regression Accuracy: 0.8661283654818763
              precision    recall  f1-score   support

           0       0.73      0.67      0.70     16181
           1       0.51      0.18      0.27      8485
           2       0.90      0.97      0.93     89025

    accuracy                           0.87    113691
   macro avg       0.71      0.61      0.63    113691
weighted avg       0.85      0.87      0.85    113691

Naive Bayes Accuracy: 0.816916026774327
              precision    recall  f1-score   support

           0       0.84      0.26      0.40     16181
           1       0.43      0.00      0.00      8485
           2       0.82      1.00      0.90     89025

    accuracy                           0.82    113691
   macro avg       0.70      0.42      0.43    113691
weighted avg       0.79      0.82      0.76    113691



In [15]:
def predict_sentiment(text, model=lr_model):
    # Preprocess
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    text = ' '.join([lemmatizer.lemmatize(w) for w in text.split()])

    # Transform
    vec = tfidf.transform([text])
    pred = model.predict(vec)[0]

    return le.inverse_transform([pred])[0]

# Example
review = "This car is amazing and very comfortable!"
print("Predicted Sentiment:", predict_sentiment(review))


Predicted Sentiment: Positive


In [17]:


# Example
review = "This toy is bad and very uncomfortable!"
print("Predicted Sentiment:", predict_sentiment(review))


Predicted Sentiment: Negative


In [18]:

# Example
review = "This car is okay!"
print("Predicted Sentiment:", predict_sentiment(review))


Predicted Sentiment: Neutral
